# Homogeneous OPT / FD Benchmark

Small 200x200 benchmarks for acoustic and elastic propagation. Acoustic compares OPT against an explicit FD5 baseline. Elastic compares OPT and a homogeneous collocated FD baseline mostly through Gaussian initial data, so the source-definition mismatch does not dominate.

## 1. Setup

In [ ]:
using Pkg
cd(@__DIR__)
Pkg.activate("../")

try
    using Metal
catch err
    @warn "Metal not loaded; continuing on CPU" err
end

using LinearAlgebra
using Statistics
using SparseArrays
using CairoMakie
CairoMakie.activate!()

ParamFile = "../config/testparam.csv"
include("../src/batchFiles/batchGPU.jl")
include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")
using .commonBatchs, .planet1D, .GeoPoints

include("../src/flexOPT.jl")
using .flexOPT

include("temporaryHelpers.jl")


## 2. Shared Grid And Source

In [ ]:
shape = (200, 200)
sourcePoint = CartesianIndex(cld(shape[1], 2), cld(shape[2], 2))
store_every = 4
Nt = 160

# Use a small physical grid spacing to keep dx/dt scales moderate and make high f0 visible.
dx = 50.0

# Acoustic background
v0 = 2600.0
velocity_homo = fill(v0, shape)
cfl_acoustic = 0.35
dt_acoustic = cfl_acoustic * dx / (sqrt(2) * v0)
delta_acoustic = (dx, dx, dt_acoustic)

# High-ish f0, still below spatial/time Nyquist diagnostics.
cfl_info_ac = cfl_diagnostics(velocity_homo, delta_acoustic; cfl_safety=cfl_acoustic, ppw=6, samples_per_period=10)
source_f0_acoustic = 0.75 * min(v0 / (6dx), 1 / (10dt_acoustic))
source_t0_acoustic = min(6 / source_f0_acoustic, 0.25Nt * dt_acoustic)
source_amplitude_acoustic = 1.0

@show shape sourcePoint Nt store_every
@show delta_acoustic cfl_info_ac source_f0_acoustic source_t0_acoustic


## 3. Acoustic: FD5 Baseline

In [ ]:
preparedFD5_ac = prepare_fd2d_acoustic_fd5_baseline(velocity_homo, delta_acoustic)
timeSignal_ac_fd = source_time_samples(Nt, dt_acoustic, preparedFD5_ac.timePointsUsedForOneStep; t0=source_t0_acoustic, f0=source_f0_acoustic)
sourceFD5_ac = point_source_full(preparedFD5_ac, sourcePoint, timeSignal_ac_fd; amplitude=source_amplitude_acoustic)

source_peak_index_ac = argmax(abs.(timeSignal_ac_fd))
source_peak_time_ac = (source_peak_index_ac - 1) * dt_acoustic
@show maximum(abs, timeSignal_ac_fd) source_peak_index_ac source_peak_time_ac maximum(abs, sourceFD5_ac)

t_fd5_ac = @elapsed frames_fd5_ac_full = propagate_linear_frames_with_source(
    preparedFD5_ac,
    Nt;
    sourceFull=sourceFD5_ac,
    store_every=store_every,
    blowup_limit=1e12,
)
frames_fd5_ac = component_frames(frames_fd5_ac_full, 1)
report_fd5_ac = wavefield_snapshot_report(frames_fd5_ac)
drift_fd5_ac = drift_report(frames_fd5_ac, sourcePoint)
@show t_fd5_ac length(frames_fd5_ac) report_fd5_ac[1] report_fd5_ac[end] drift_fd5_ac[end]


In [ ]:
fig_fd5_ac = plot_wave_snapshots(
    frames_fd5_ac;
    sourcePoint=sourcePoint,
    title="Acoustic homogeneous FD5, point source",
)
fig_fd5_ac


## 4. Acoustic: OPT Source Benchmark

In [ ]:
# Acoustic OPT in physical small-delta coordinates.
# If constructAmatrix becomes ill-scaled again, switch to velocity_cfl + delta=(1,1,1), as in propagateInside_run.
opt_ac = build_opt_prepared(
    "2DacousticTime",
    [velocity_homo],
    delta_acoustic;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="homogeneous_200_acoustic_OPT",
)
preparedOPT_ac = opt_ac.prepared
st_opt_ac = operator_stencil_at_point(opt_ac.numOps, sourcePoint; which=:left)
st_rhs_ac = rhs_stencil_at_source(opt_ac.numOps, sourcePoint; iExpr=1, iForceField=1)
@show implicit_matrix_report(preparedOPT_ac)
stencil_time_summary(st_opt_ac), stencil_matrices_by_time(st_opt_ac), stencil_time_summary(st_rhs_ac), stencil_matrices_by_time(st_rhs_ac)


In [ ]:
timeSignal_ac_opt = source_time_samples(Nt, dt_acoustic, preparedOPT_ac.timePointsUsedForOneStep; t0=source_t0_acoustic, f0=source_f0_acoustic)
sourceOPT_ac = point_source_full(preparedOPT_ac, sourcePoint, timeSignal_ac_opt; amplitude=source_amplitude_acoustic)
source_rhs_scan_ac = source_rhs_scan(preparedOPT_ac, sourceOPT_ac; its=1:min(12, Nt))
source_rhs_peak_ac = source_rhs_diagnostics(preparedOPT_ac, sourceOPT_ac; it=min(argmax(abs.(timeSignal_ac_opt)), Nt))
@show maximum(abs, sourceOPT_ac) source_rhs_scan_ac
@show source_rhs_peak_ac.b_max source_rhs_peak_ac.b_norm source_rhs_peak_ac.b_argmax

t_opt_ac = @elapsed frames_opt_ac_full = propagate_linear_frames_with_source(
    preparedOPT_ac,
    Nt;
    sourceFull=sourceOPT_ac,
    store_every=store_every,
    blowup_limit=1e12,
)
frames_opt_ac = component_frames(frames_opt_ac_full, 1)
report_opt_ac = wavefield_snapshot_report(frames_opt_ac)
drift_opt_ac = drift_report(frames_opt_ac, sourcePoint)
@show t_opt_ac length(frames_opt_ac) report_opt_ac[1] report_opt_ac[end] drift_opt_ac[end]


In [ ]:
fig_opt_ac = plot_wave_snapshots(
    frames_opt_ac;
    sourcePoint=sourcePoint,
    title="Acoustic homogeneous OPT, point source",
)
fig_opt_ac


## 5. Acoustic LHS Benchmark From Gaussian Initial Field

This avoids source differences and compares only propagation from the same initial condition.

In [ ]:
initial_ac = gaussian_field(shape, sourcePoint; sigma=8.0, amplitude=1.0)
zero_ac = zeros(Float64, shape)

t_fd5_gauss_ac = @elapsed frames_fd5_gauss_ac_full = propagate_linear_frames_with_source(
    preparedFD5_ac,
    Nt;
    initialPast=initial_ac,
    initialPresent=initial_ac,
    store_every=store_every,
    blowup_limit=1e12,
)
t_opt_gauss_ac = @elapsed frames_opt_gauss_ac_full = propagate_linear_frames_with_source(
    preparedOPT_ac,
    Nt;
    initialPast=initial_ac,
    initialPresent=initial_ac,
    store_every=store_every,
    blowup_limit=1e12,
)
frames_fd5_gauss_ac = component_frames(frames_fd5_gauss_ac_full, 1)
frames_opt_gauss_ac = component_frames(frames_opt_gauss_ac_full, 1)
diff_gauss_ac = frame_difference_report(frames_opt_gauss_ac, frames_fd5_gauss_ac)
@show t_fd5_gauss_ac t_opt_gauss_ac diff_gauss_ac[end]


In [ ]:
fig_gauss_ac = Figure(size=(1100, 450))
idx = length(frames_fd5_gauss_ac)
clim = maximum(abs, frames_fd5_gauss_ac[idx])
ax1 = Axis(fig_gauss_ac[1, 1], aspect=DataAspect(), title="FD5 acoustic Gaussian")
heatmap!(ax1, frames_fd5_gauss_ac[idx]; colormap=:balance, colorrange=(-clim, clim))
ax2 = Axis(fig_gauss_ac[1, 2], aspect=DataAspect(), title="OPT acoustic Gaussian")
heatmap!(ax2, frames_opt_gauss_ac[idx]; colormap=:balance, colorrange=(-clim, clim))
fig_gauss_ac


## 6. Elastic Homogeneous: FD Baseline And OPT

FD here is a homogeneous collocated displacement baseline with body-force RHS. The clean comparison below uses Gaussian initial displacement, not source injection.

In [ ]:
rho0 = 2500.0
vp0 = 3200.0
vs0 = 1800.0
rho_el = fill(rho0, shape)
vp_el = fill(vp0, shape)
vs_el = fill(vs0, shape)
mu_el = rho_el .* vs_el.^2
lambda_el = rho_el .* vp_el.^2 .- 2 .* mu_el
models_el = [rho_el, lambda_el, mu_el]

cfl_elastic = 0.25
dt_elastic = cfl_elastic * dx / (sqrt(2) * vp0)
delta_elastic = (dx, dx, dt_elastic)
Nt_elastic = 120
store_every_elastic = 4
@show delta_elastic Nt_elastic extrema(lambda_el) extrema(mu_el)

preparedFD_el = prepare_fd2d_elastic_homogeneous_baseline(rho_el, lambda_el, mu_el, delta_elastic)
opt_el = build_opt_prepared(
    "2DsismoTimeIsoHetero",
    models_el,
    delta_elastic;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="homogeneous_200_elastic_OPT",
)
preparedOPT_el = opt_el.prepared
@show implicit_matrix_report(preparedFD_el)
@show implicit_matrix_report(preparedOPT_el)


In [ ]:
initial_ux = gaussian_field(shape, sourcePoint; sigma=10.0, amplitude=1.0)
initial_uz = zeros(Float64, shape)
initial_el = zeros(Float64, shape..., 2)
initial_el[:, :, 1] .= initial_ux
initial_el[:, :, 2] .= initial_uz

t_fd_el = @elapsed frames_fd_el_full = propagate_linear_frames_with_source(
    preparedFD_el,
    Nt_elastic;
    initialPast=initial_el,
    initialPresent=initial_el,
    store_every=store_every_elastic,
    blowup_limit=1e12,
)
t_opt_el = @elapsed frames_opt_el_full = propagate_linear_frames_with_source(
    preparedOPT_el,
    Nt_elastic;
    initialPast=initial_el,
    initialPresent=initial_el,
    store_every=store_every_elastic,
    blowup_limit=1e12,
)

ux_fd_el = component_frames(frames_fd_el_full, 1)
uz_fd_el = component_frames(frames_fd_el_full, 2)
ux_opt_el = component_frames(frames_opt_el_full, 1)
uz_opt_el = component_frames(frames_opt_el_full, 2)
diff_ux_el = frame_difference_report(ux_opt_el, ux_fd_el)
diff_uz_el = frame_difference_report(uz_opt_el, uz_fd_el)
@show t_fd_el t_opt_el
@show wavefield_snapshot_report(ux_fd_el)[end] wavefield_snapshot_report(ux_opt_el)[end]
@show diff_ux_el[end] diff_uz_el[end]


In [ ]:
fig_el = Figure(size=(1100, 850))
idx = length(ux_fd_el)
clim_ux = maximum(abs, ux_fd_el[idx])
clim_uz = max(maximum(abs, uz_fd_el[idx]), maximum(abs, uz_opt_el[idx]), eps(Float64))
ax11 = Axis(fig_el[1, 1], aspect=DataAspect(), title="FD elastic ux")
heatmap!(ax11, ux_fd_el[idx]; colormap=:balance, colorrange=(-clim_ux, clim_ux))
ax12 = Axis(fig_el[1, 2], aspect=DataAspect(), title="OPT elastic ux")
heatmap!(ax12, ux_opt_el[idx]; colormap=:balance, colorrange=(-clim_ux, clim_ux))
ax21 = Axis(fig_el[2, 1], aspect=DataAspect(), title="FD elastic uz")
heatmap!(ax21, uz_fd_el[idx]; colormap=:balance, colorrange=(-clim_uz, clim_uz))
ax22 = Axis(fig_el[2, 2], aspect=DataAspect(), title="OPT elastic uz")
heatmap!(ax22, uz_opt_el[idx]; colormap=:balance, colorrange=(-clim_uz, clim_uz))
fig_el


## 7. Marmousi Small-Delta Sketch

This section is intentionally a template: resample/crop Marmousi to a smaller `dx` if the data resolution permits it. The acoustic check above suggests that trusting wavefields at smaller `δ` is the right direction.

In [ ]:
# marmousi = load(joinpath(@__DIR__, "tmp/seismicModelMarmousi.jld2"), "output")
# shape_m = (200, 200)
# downsample_step_m = 1
# velocity_m = downsample_center_crop(marmousi.Vpv .* 1e3, shape_m; step=downsample_step_m)
# dx_m = 100.0 # set to the true data spacing if known
# cfl_m = 0.35
# dt_m = cfl_m * dx_m / (sqrt(2) * maximum(velocity_m))
# delta_m = (dx_m, dx_m, dt_m)
# preparedFD5_m = prepare_fd2d_acoustic_fd5_baseline(velocity_m, delta_m)
# opt_m = build_opt_prepared("2DacousticTime", [velocity_m], delta_m; pointsInSpace=3, pointsInTime=3, supplementaryOrder=2, YorderBspace=-1, YorderBtime=-1)
